# Overview of Transformer Models: Text Generation Models

The most common picture of understanding the behavior of a Transformer LLM is to think of it as a software system that takes in text and generates text in response. Once a large enough text-in-text-out model is trained on a large enough high-quality dataset, it becomes able to generate impressive and useful outputs.

The model does not generate the text all in one operation; it actually generates one token at a time. Each token generation step is one forward pass through the model (that’s machine-learning speak for the inputs going into the neural network and flowing through the computations it needs to produce an output on the other end of the computation graph). After each token generation, we tweak the input prompt for the next generation step by appending the output token to the end of the input prompt. 

There’s a specific word used in machine learning to describe models that consume their earlier predictions to make later predictions (e.g., the model’s first generated token is used to generate the second token). They’re called autoregressive models. That is why you’ll hear text generation LLMs being called autoregressive models. This is often used to differentiate text generation models from text representation models like BERT, which are not autoregressive.

In [3]:
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", 
    device_map=device,
    torch_dtype='auto', 
    trust_remote_code=True, 
)

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False, 
    max_new_tokens=50, 
    do_sample=False
    
)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
output = generator(prompt)

In [7]:
print(output[0]['generated_text'])

 Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


We can see the model begin to write the email starting with the subject. It stopped abruptly because it reached the token limit we established by setting max_new_tokens to 50 tokens.

## Model Layers

The flow of the computation follows the direction of the arrow from top to bottom. For each generated token, the process flows once through each of the Transformer blocks in the stack in order, then to the LM head, which finally outputs the probability distribution for the next token,

![Alt text](figures/transformer-flow.png)

Each of the token streams starts with an input vector (the embedding vector and some positional information; we’ll discuss positional embeddings later in the chapter). At the end of the stream, another vector emerges as the result of the model’s processing,

![Alt text](figures/transformer-flow-2.png)

Recall from Chapter 2 that the tokenizer contains a table of tokens—the tokenizer’s vocabulary. The model has a vector representation associated with each of these tokens in the vocabulary (token embeddings).We can display the order of the layers by simply printing out the model variable. For this model, we have:

In [11]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206

- This shows us the various nested layers of the model. The majority of the model is labeled model, followed by lm_head.
- Inside the Phi3Model model, we see the embeddings matrix embed_tokens and its dimensions. It has 32,064 tokens each with a vector size of 3,072.
- Skipping the dropout layer for now, we can see the next major component is the stack of Transformer decoder layers. It contains 32 blocks of type Phi3DecoderLayer.
- Each of these Transformer blocks includes an attention layer and a feedforward neural network (also known as an mlp or multilevel perceptron). We’ll cover these in more detail later in the chapter.
- Finally, we see the lm_head taking a vector of size 3,072 and outputting a vector equivalent to the number of tokens the model knows. That output is the probability score for each token that helps us select the output token.


## Sampling/Decoding - Choosing a Single Token from the Probability Distribution
At the end of processing, the output of the model is a probability score for each token in the vocabulary. The method of choosing a single token from the probability distribution is called the decoding strategy. The easiest decoding strategy would be to always pick the token with the highest probability score. In practice, this doesn’t tend to lead to the best outputs for most use cases. A better approach is to add some randomness and sometimes choose the second or third highest probability token. The idea here is to basically sample from the probability distribution based on the probability score, as the statisticians would say. What this means is that if the token “Dear” has a 40% probability of being the next token, then it has a 40% chance of being picked (instead of greedy search, which would pick it directly for having the highest score). So with this method, all the other tokens have a chance of being picked according to their score.

Example of showing probability below, 

In [15]:
prompt = "The capital of France is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids= input_ids.to("mps")

# Get the output of the model before the lm head 
model_output = model.model(input_ids)

# Get the output of the lm head
lm_head_output = model.lm_head(model_output[0])
print(lm_head_output)
print(lm_head_output.shape)

tensor([[[25.0000, 25.1250, 23.0000,  ..., 19.1250, 19.1250, 19.1250],
         [31.0000, 31.5000, 26.0000,  ..., 25.8750, 25.8750, 25.8750],
         [31.5000, 28.8750, 31.0000,  ..., 26.2500, 26.2500, 26.2500],
         [33.0000, 31.8750, 36.0000,  ..., 27.8750, 27.8750, 27.8750],
         [27.8750, 29.5000, 28.2500,  ..., 20.5000, 20.5000, 20.5000]]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<LinearBackward0>)
torch.Size([1, 5, 32064])


Now, `lm_head_output` is of the shape [1, 6, 32064]. We can access the token probability scores for the last generated token using `lm_head_output[0,-1]`, which uses the index 0 across the batch dimension; the index –1 gets us the last token in the sequence. This is now a list of probability scores for all 32,064 tokens. We can get the top scoring token ID, and then decode it to arrive at the text of the generated output token:

In [17]:
lm_head_output[0, -1]

tensor([27.8750, 29.5000, 28.2500,  ..., 20.5000, 20.5000, 20.5000],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [18]:
token_id = lm_head_output[0, -1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

Recall that the output of `lm_head` was of the shape `[1, 5, 32064]`. That was because the input to it was of the shape `[1, 5, 3072]`, which is a batch of one input string, containing six tokens, each of them represented by a vector of size 3,072 corresponding to the output vectors after the stack of Transformer blocks.

In [22]:
print(f"Model output shape before LM Head: {model_output[0].shape}")
print(f"Output shape of LM Head: {lm_head_output.shape}")

Model output shape before LM Head: torch.Size([1, 5, 3072])
Output shape of LM Head: torch.Size([1, 5, 32064])


## Speeding up Generation by Caching Keys and Values

Recall that when generating the second token, we simply append the output token to the input and do another forward pass through the model. If we give the model the ability to cache the results of the previous calculation (especially some of the specific vectors in the attention mechanism), we no longer need to repeat the calculations of the previous streams. This time the only needed calculation is for the last stream. This is an optimization technique called the keys and values (kv) cache and it provides a significant speedup of the generation process. Keys and values are some of the central components of the attention mechanism, as we’ll see later in this chapter.

In [4]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors='pt').to('mps')

In [29]:
%%timeit -n 1

# Generate the text
generation_output = model.generate(
    input_ids=input_ids['input_ids'], 
    attention_mask=input_ids['attention_mask'],
    max_new_tokens = 100,
    use_cache=True
)

The slowest run took 4.08 times longer than the fastest. This could mean that an intermediate result is being cached.
49.2 s ± 29.8 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


How long would it take if we disable the cache

In [5]:
%%timeit -n 1

# Generate the text
generation_output = model.generate(
    input_ids=input_ids['input_ids'], 
    attention_mask=input_ids['attention_mask'],
    max_new_tokens = 100,
    use_cache=False
)

You are not running the flash-attention implementation, expect numerical differences.


The slowest run took 5.99 times longer than the fastest. This could mean that an intermediate result is being cached.
3min 38s ± 2min 13s per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Inside the Transformer Block
A Transformer block is made up of two successive components:

![Alt text](figures/transformer-block.png)

The attention layer is mainly concerned with incorporating relevant information from other input tokens and positions
The feedforward layer houses the majority of the model’s processing capacity.

Two main steps are involved in the attention mechanism:

1. A way to score how relevant each of the previous input tokens are to the current token being processed.
2. Using those scores, we combine the information from the various positions into a single output vector.

**Multi-Head Attention**: 
To give the Transformer more extensive attention capability, the attention mechanism is duplicated and executed multiple times in parallel. Each of these parallel applications of attention is conducted into an attention head. This increases the model’s capacity to model complex patterns in the input sequence that require paying attention to different patterns at once.

**How self-attention is calculated**

Let’s look at how attention is calculated inside a single attention head. Before we start the calculation, let’s observe the following as the starting position:

- The attention layer (of a generative LLM) is processing attention for a single position.
- The inputs to the layer are:
  - The vector representation of the current position or token
  - The vector representations of the previous tokens
- The goal is to produce a new representation of the current position that incorporates relevant information from the previous tokens:
  - For example, if we’re processing the last position in the sentence “Sarah fed the cat because it,” we want “it” to represent the cat—so attention bakes in “cat information” from the cat token.
- The training process produces three projection matrices that produce the components that interact in this calculation:
  - A query projection matrix
  - A key projection matrix
  - A value projection matrix

![Alt text](figures/Attention-head.png)

Attention starts by multiplying the inputs by the projection matrices to create three new matrices. These are called the queries, keys, and values matrices. These matrices contain the information of the input tokens projected to three different spaces that help carry out the two steps of attention:

- Relevance scoring
- Combining information

Below shows these three new matrices, and how the bottom row of all three matrices is associated with the current position while the rows above it are associated with the previous positions.

![Alt text](figures/attention-matrices.png)

**Self-attention: Relevance Scoring**

The relevance scoring step of attention is conducted by multiplying the query vector of the current position with the keys matrix. This produces a score stating how relevant each previous token is. Passing that by a softmax operation normalizes these scores so they sum up to 1. Shown below, 

![Alt text](figures/attention-relevant-scores.png)

**Self-attention: Combining Information**

Now that we have the relevance scores, we multiply the value vector associated with each token by that token’s score. Summing up those resulting vectors produces the output of this attention step, as we see below. 

![Alt text](figures/attention-combining-info.png)


## Detailed Transformers Block

 Recall that the two major components of a Transformer block are an attention layer and a feedforward neural network. A more detailed view of the block would also reveal the residual connections and layer-normalization operations that we can see below, 

![Alt text](figures/detailed-transformer-block.png)

Positional embeddings have been a key component since the original Transformer. They enable the model to keep track of the order of tokens/words in a sequence/sentence, which is an indispensable source of information in language.
The original Transformer paper and some of the early variants had absolute positional embeddings that, in essence, marked the first token as position 1, the second as position 2...etc. These could either be static methods (where the positional vectors are generated using geometric functions) or learned (where the model training assigns them their values during the learning process).

## Essentially

- A Transformer LLM generates one token at a time.
- That output token is appended to the prompt, then this updated prompt is presented to the model again for another forward pass to generate the next token.
- The three major components of the Transformer LLM are the tokenizer, a stack of Transformer blocks, and a language modeling head.
- The tokenizer contains the token vocabulary for the model. The model has token embeddings associated with those tokens. Breaking the text into tokens and then using the embeddings of these tokens is the first step in the token generation process.
- The forward pass flows through all the stages once, one by one.
- Near the end of the process, the LM head scores the probabilities of the next possible token. Decoding strategies inform which actual token to pick as the output for this generation step (sometimes it’s the most probable next token, but not always).
- One reason the Transformer excels is its ability to process tokens in parallel. Each of the input tokens flow into their individual tracks or streams of processing. The number of streams is the model’s “context size” and this represents the max number of tokens the model can operate on.
- Because Transformer LLMs loop to generate the text one token at a time, it’s a good idea to cache the processing results of each step so we don’t duplicate the processing effort (these results are stored as various matrices within the layers).
- The majority of processing happens within Transformer blocks. These are made up of two components. One of them is the feedforward neural network, which is able to store information and make predictions and interpolations from data it was trained on.
- The second major component of a Transformer block is the attention layer. Attention incorporates contextual information to allow the model to better capture the nuance of language.
- Attention happens in two major steps: (1) scoring relevance and (2) combining information.
- A Transformer attention layer conducts several attention operations in parallel, each occurring inside an attention head, and their outputs are aggregated to make up the output of the attention layer.
- Attention can be accelerated via sharing the keys and values matrices between all heads, or groups of heads (grouped-query attention).
- Methods like Flash Attention speed up the attention calculation by optimizing how the operation is done on the different memory systems of a GPU.
